# Phishing Website Detection

This notebook builds a machine learning model to detect phishing websites.

**Goal:** Classify URLs as either:
- 0 = Legitimate
- 1 = Phishing

**Models Used:**
- Logistic Regression
- Random Forest

## 1. Import Required Libraries

In [ ]:
# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for plotting
import matplotlib.pyplot as plt

# Import seaborn for statistical visualizations
import seaborn as sns

# Import machine learning tools from scikit-learn
from sklearn.model_selection import train_test_split  # For splitting data
from sklearn.preprocessing import StandardScaler      # For feature scaling
from sklearn.linear_model import LogisticRegression  # Logistic Regression model
from sklearn.ensemble import RandomForestClassifier  # Random Forest model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix  # Evaluation metrics

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set random seed for reproducibility
RANDOM_STATE = 42

## 2. Load the Dataset

In [ ]:
# Load the dataset from CSV file
df = pd.read_csv('data/dataset.csv')

# Display the first few rows
print("First 5 rows of the dataset:")
df.head()

## 3. Data Inspection

In [ ]:
# Check the shape of the dataset (rows, columns)
print(f"Dataset shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Check column names
print("Column names:")
print(df.columns.tolist())

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
# Check class distribution (target variable)
print("Class distribution:")
print(df['label'].value_counts())
print(f"\nClass percentages:")
print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='label', data=df)
plt.title('Class Distribution (0=Legitimate, 1=Phishing)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

## 4. Data Cleaning

In [ ]:
# Remove duplicate rows if any
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")
    print(f"New dataset shape: {df.shape}")
else:
    print("No duplicates to remove")

In [ ]:
# Remove rows with missing values if any
missing_before = df.isnull().sum().sum()
if missing_before > 0:
    df = df.dropna()
    print(f"Removed {missing_before} rows with missing values")
    print(f"New dataset shape: {df.shape}")
else:
    print("No missing values to remove")

## 5. Feature Engineering

We'll extract simple features from URLs that can help distinguish phishing from legitimate sites.

In [ ]:
def extract_url_features(url):
    """
    Extract features from a URL.
    
    Args:
        url: String containing the URL
    
    Returns:
        Dictionary of extracted features
    """
    features = {}
    
    # Convert to string if not already
    url = str(url)
    
    # 1. URL length
    features['url_length'] = len(url)
    
    # 2. Number of dots in URL
    features['num_dots'] = url.count('.')
    
    # 3. Number of hyphens in URL
    features['num_hyphens'] = url.count('-')
    
    # 4. Number of special characters
    special_chars = "!@#$%^&*()_+=[]{}|;:'\"<>,/?"
    features['num_special_chars'] = sum(1 for char in url if char in special_chars)
    
    # 5. Number of digits in URL
    features['num_digits'] = sum(char.isdigit() for char in url)
    
    # 6. Presence of HTTPS (1 if yes, 0 if no)
    features['has_https'] = 1 if url.startswith('https://') else 0
    
    # 7. Presence of IP address (basic check for numeric patterns)
    # Check if URL contains pattern like xxx.xxx.xxx.xxx
    import re
    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    features['has_ip_address'] = 1 if re.search(ip_pattern, url) else 0
    
    # 8. Number of subdomains (count dots in domain part)
    # Remove protocol first
    domain_part = url.replace('https://', '').replace('http://', '').split('/')[0]
    features['num_subdomains'] = domain_part.count('.')
    
    # 9. Presence of suspicious words
    suspicious_words = ['login', 'verify', 'secure', 'account', 'bank', 'update', 'confirm', 'password']
    features['has_suspicious_words'] = 1 if any(word in url.lower() for word in suspicious_words) else 0
    
    return features

# Test the feature extraction function on a sample URL
sample_url = "http://secure-login-paypal.com"
print(f"Testing feature extraction on: {sample_url}")
print(extract_url_features(sample_url))

In [ ]:
# Apply feature extraction to all URLs in the dataset
print("Extracting features from all URLs...")

# Create a list of feature dictionaries
features_list = df['url'].apply(extract_url_features)

# Convert to DataFrame
features_df = pd.DataFrame(features_list.tolist())

# Display the first few rows of extracted features
print("Extracted features:")
features_df.head()

In [ ]:
# Combine features with the original dataset
df_final = pd.concat([df, features_df], axis=1)

# Display the final dataset
print("Final dataset with features:")
df_final.head()

## 6. Prepare Features and Target

In [ ]:
# Separate features (X) and target (y)
# X = all feature columns (except 'url' and 'label')
# y = target column ('label')

feature_columns = ['url_length', 'num_dots', 'num_hyphens', 'num_special_chars', 
                   'num_digits', 'has_https', 'has_ip_address', 'num_subdomains', 'has_suspicious_words']

X = df_final[feature_columns]
y = df_final['label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {feature_columns}")

## 7. Train/Test Split

We split the data into training (80%) and testing (20%) sets. This is crucial to evaluate how well the model performs on unseen data.

In [ ]:
# Split the data into training and testing sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state ensures reproducibility
# stratify=y ensures the class distribution is preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts(normalize=True) * 100)
print(f"\nTesting class distribution:")
print(y_test.value_counts(normalize=True) * 100)

## 8. Feature Scaling

Scale the features to have similar ranges. This helps some algorithms perform better.

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on training data only (prevents data leakage)
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the fitted scaler
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")
print(f"Scaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")

## 9. Train Logistic Regression Model

In [ ]:
# Initialize Logistic Regression model
# max_iter=1000 ensures the algorithm has enough iterations to converge
logistic_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)

# Train the model on the scaled training data
logistic_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

## 10. Train Random Forest Model

In [ ]:
# Initialize Random Forest model
# n_estimators=100 means 100 decision trees in the forest
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Train the model on the scaled training data
# Note: Random Forest doesn't require scaled features, but we use scaled data for consistency
random_forest_model.fit(X_train_scaled, y_train)

print("Random Forest model trained successfully.")

## 11. Evaluate Logistic Regression Model

In [ ]:
# Make predictions on the test set
y_pred_logistic = logistic_model.predict(X_test_scaled)

# Calculate evaluation metrics
logistic_accuracy = accuracy_score(y_test, y_pred_logistic)
logistic_precision = precision_score(y_test, y_pred_logistic)
logistic_recall = recall_score(y_test, y_pred_logistic)
logistic_f1 = f1_score(y_test, y_pred_logistic)
logistic_cm = confusion_matrix(y_test, y_pred_logistic)

print("Logistic Regression Model Evaluation:")
print(f"Accuracy: {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall: {logistic_recall:.4f}")
print(f"F1-Score: {logistic_f1:.4f}")
print(f"\nConfusion Matrix:")
print(logistic_cm)

In [ ]:
# Visualize the confusion matrix for Logistic Regression
plt.figure(figsize=(8, 6))
sns.heatmap(logistic_cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 12. Evaluate Random Forest Model

In [ ]:
# Make predictions on the test set
y_pred_rf = random_forest_model.predict(X_test_scaled)

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Model Evaluation:")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall: {rf_recall:.4f}")
print(f"F1-Score: {rf_f1:.4f}")
print(f"\nConfusion Matrix:")
print(rf_cm)

In [ ]:
# Visualize the confusion matrix for Random Forest
plt.figure(figsize=(8, 6))
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 13. Model Comparison

In [ ]:
# Create a comparison table
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [logistic_accuracy, rf_accuracy],
    'Precision': [logistic_precision, rf_precision],
    'Recall': [logistic_recall, rf_recall],
    'F1-Score': [logistic_f1, rf_f1]
})

# Display the comparison table
print("Model Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize the comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
logistic_scores = [logistic_accuracy, logistic_precision, logistic_recall, logistic_f1]
rf_scores = [rf_accuracy, rf_precision, rf_recall, rf_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, logistic_scores, width, label='Logistic Regression')
bars2 = ax.bar(x + width/2, rf_scores, width, label='Random Forest')

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1.1])

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 14. Select the Best Model

In [ ]:
# Select the best model based on F1-Score (balance of precision and recall)
# For phishing detection, we want to balance catching phishing sites (recall) with not flagging legitimate sites (precision)

if rf_f1 > logistic_f1:
    best_model = random_forest_model
    best_model_name = "Random Forest"
    best_f1 = rf_f1
else:
    best_model = logistic_model
    best_model_name = "Logistic Regression"
    best_f1 = logistic_f1

print(f"Best model: {best_model_name}")
print(f"Best F1-Score: {best_f1:.4f}")

## 15. Feature Importance (Random Forest)

In [ ]:
# Get feature importance from Random Forest model
if best_model_name == "Random Forest":
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': random_forest_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("Feature Importance (Random Forest):")
    print(feature_importance.to_string(index=False))
    
    # Visualize feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_importance)
    plt.title('Feature Importance - Random Forest')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.show()

## 16. Create Prediction Function

In [ ]:
def predict_url(url, model=best_model, scaler=scaler, feature_columns=feature_columns):
    """
    Predict whether a URL is phishing or legitimate.
    
    Args:
        url: The URL to classify
        model: The trained model to use for prediction
        scaler: The fitted scaler for feature scaling
        feature_columns: List of feature column names
    
    Returns:
        String: 'Phishing' or 'Legitimate'
    """
    # Extract features from the URL
    features = extract_url_features(url)
    
    # Convert to DataFrame in the correct order
    features_df = pd.DataFrame([features])[feature_columns]
    
    # Scale the features
    features_scaled = scaler.transform(features_df)
    
    # Make prediction
    prediction = model.predict(features_scaled)[0]
    
    # Return the result
    if prediction == 1:
        return "Phishing"
    else:
        return "Legitimate"

print("Prediction function created successfully.")

## 17. Test the Prediction Function

In [ ]:
# Test URLs - mix of legitimate and phishing examples
test_urls = [
    # Legitimate URLs
    "https://www.google.com",
    "https://www.facebook.com",
    "https://www.amazon.com",
    "https://www.linkedin.com",
    "https://github.com/user/project",
    
    # Phishing URLs
    "http://secure-login-paypal.com",
    "http://apple-id-verify.com",
    "http://netflix-account-verify.com",
    "http://facebook-security-check.com",
    "http://bit.ly/secure-login"
]

print("Testing prediction function on sample URLs:")
print("=" * 60)

for url in test_urls:
    prediction = predict_url(url)
    print(f"URL: {url}")
    print(f"Prediction: {prediction}")
    print("-" * 60)

## 18. Summary and Results

In [ ]:
print("=" * 70)
print("PHISHING WEBSITE DETECTION - PROJECT SUMMARY")
print("=" * 70)
print(f"\nDataset Size: {df_final.shape[0]} URLs")
print(f"Features Used: {len(feature_columns)}")
print(f"Feature Columns: {feature_columns}")
print(f"\nModels Trained: Logistic Regression, Random Forest")
print(f"Best Model: {best_model_name}")
print(f"\nFinal Model Performance ({best_model_name}):")
if best_model_name == "Random Forest":
    print(f"  Accuracy: {rf_accuracy:.4f}")
    print(f"  Precision: {rf_precision:.4f}")
    print(f"  Recall: {rf_recall:.4f}")
    print(f"  F1-Score: {rf_f1:.4f}")
else:
    print(f"  Accuracy: {logistic_accuracy:.4f}")
    print(f"  Precision: {logistic_precision:.4f}")
    print(f"  Recall: {logistic_recall:.4f}")
    print(f"  F1-Score: {logistic_f1:.4f}")
print("=" * 70)